In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

In [19]:
from pathlib import Path
import pandas as pd
import os
import sklearn
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import PIL
from PIL import Image

In [20]:
print("Pandas Version:", pd.__version__)
print("Scikit-Learn Version", sklearn.__version__)
print("Tensorflow Version", tf.__version__)
print("PIL Version:", PIL.__version__)

Pandas Version: 2.2.2
Scikit-Learn Version 1.6.1
Tensorflow Version 2.20.0
PIL Version: 11.3.0


In [5]:
#Current Working Path
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd()

#dataset root
DATA_DIR = Path("/content/drive/MyDrive/Datasets/Image Dataset/data")

print("Train exists:", os.path.exists(os.path.join(DATA_DIR, "train")))
print("Test exists:", os.path.exists(os.path.join(DATA_DIR, "test")))
print("Train csv exists:", os.path.exists(os.path.join(DATA_DIR, "Training_set.csv")))
print("Test csv exists:", os.path.exists(os.path.join(DATA_DIR, "Testing_set.csv")))

Train exists: True
Test exists: True
Train csv exists: True
Test csv exists: True


In [6]:
#load the training csv dataset & display the 1st 5 rows
train_df = pd.read_csv(DATA_DIR / "Training_set.csv")
train_df.head()

,filename,label
0,Image_1.jpg,SOUTHERN DOGFACE
1,Image_2.jpg,ADONIS
2,Image_3.jpg,BROWN SIPROETA
3,Image_4.jpg,MONARCH
4,Image_5.jpg,GREEN CELLED CATTLEHEART


In [7]:
train_df['image_path'] = train_df['filename'].apply(
    lambda x: os.path.join(DATA_DIR, "train", x)
)

In [8]:
train_df.sample(5)

,filename,label,image_path
887,Image_888.jpg,RED CRACKER,/content/drive/MyDrive/Datasets/Image Dataset/...
6011,Image_6012.jpg,ADONIS,/content/drive/MyDrive/Datasets/Image Dataset/...
6355,Image_6356.jpg,SOOTYWING,/content/drive/MyDrive/Datasets/Image Dataset/...
3221,Image_3222.jpg,CHESTNUT,/content/drive/MyDrive/Datasets/Image Dataset/...
6106,Image_6107.jpg,COMMON BANDED AWL,/content/drive/MyDrive/Datasets/Image Dataset/...


In [9]:
#checking NAN values
train_df.isna().sum()

,0
filename,0
label,0
image_path,0


In [10]:
#checking duplicates
train_df.duplicated().sum()

np.int64(0)

In [11]:
# Check the distribution of classes in the target column
train_df['label'].value_counts()

,count
label,
MOURNING CLOAK,131
SLEEPY ORANGE,107
ATALA,100
BROWN SIPROETA,99
SCARCE SWALLOW,97
...,...
AMERICAN SNOOT,74
GOLD BANDED,73
MALACHITE,73


In [12]:
# Print the minimum and maximum class frequencies for label column
print(f"Minimum frequency: {train_df['label'].value_counts().min()}")
print(f"Maximum frequency: {train_df['label'].value_counts().max()}")

Minimum frequency: 71
Maximum frequency: 131


Sufficient images in each class; Ok for training

In [13]:
# Check for any missing images by verifying if the paths exist
missing_files = train_df[~train_df['image_path'].apply(os.path.exists)]
print("Missing Count:",len(missing_files))

Missing Count: 0


In [14]:
#label encoding for label column
label_en = LabelEncoder()
train_df['encoded'] = label_en.fit_transform(train_df['label'])

In [15]:
train_df[['label', 'encoded']].head()

,label,encoded
0,SOUTHERN DOGFACE,66
1,ADONIS,0
2,BROWN SIPROETA,12
3,MONARCH,44
4,GREEN CELLED CATTLEHEART,33


# Now Real EDA start

In [18]:
#File formate check
valid_ext = (".jpg", ".jpeg", ".png")
invalid_files = []

for path in train_df['image_path']:
    if not path.lower().endswith(valid_ext):
        invalid_files.append(path)

print("Invalid formate images:", len(invalid_files))

Invalid formate images: 0


In [ ]:
#RGB/ corruted image check
corrupted = []
non_rgb = []

for path in train_df['image_path']:
    try:
        img = Image.open(path)

        #check RGB
        if img.mode != "RGB":
            non_rgb.append(path)

            img.verify()
    except:
        corrupted.append(path)

print("Corrupted Images:", len(corrupted))
print("Non RGB:", len(non_rgb))

In [16]:
def load_image(image_path, label):

    #read image file
    img = tf.io.read_file(image_path)
    #decode image
    img = tf.image.decode_jpeg(img, channels=3)
    #Resize image
    img = tf.image.resize(img, (244, 244))
    #normalize
    img = img / 255

    return img, label

This function handles the image preprocessing pipeline for the TensorFlow:

1.Load & Decode:Reads raw image files from disk and converts them into 3-channel (RGB) tensors.
2.Resize: Standardizes all images to a uniform size of 244 X 244 pixels, as required by the model.
3.Normalize:Scales pixel values from [0, 255] down to [0, 1] to ensure faster and more stable model training.

In short, it transforms raw image files into optimized mathematical tensors ready for deep learning training.

In [17]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["encoded"].values)
)

train_ds = train_ds.map(load_image)

train_ds = train_ds.shuffle(1000)

train_ds = train_ds.batch(32)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)